In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

module_path = os.path.abspath(os.path.join("../.."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
from machines.predictions import RF_Prediction
from preprocesses.preprocess import Preprocess
from preprocesses.enums import TransformEnum, TypeFileEnum, StandardScaleEnum
from metrics.error_metric import ErrorMetric
from metrics.enums import MetricEnum
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
file_csv = "../../../data/obregon_1516_1617/original/o_57_o_phenotypic_weather_d.csv"
# print(os.getcwd())
df = pd.read_csv(file_csv)

columns_no_ndvi = []
for col in df.columns:
    if "ndvi" not in col.lower():
        columns_no_ndvi.append(col)

print(len(columns_no_ndvi))
new_csv_file = "test_no_ndvi.csv"
df.to_csv(new_csv_file, index=False)
print(columns_no_ndvi)

2054
['TGW', 'HI', 'BM', 'CT_UAV_1', 'CT_UAV_2', 'CT_UAV_3', 'CT_UAV_4', '20151213_TempOut_AVG', '20151213_HiTemp_AVG', '20151213_LowTemp_AVG', '20151213_OutHum_AVG', '20151213_DewPt_AVG', '20151213_WindSpeed_AVG', '20151213_WindRun_AVG', '20151213_HiSpeed_AVG', '20151213_WindChill_AVG', '20151213_HeatIndex_AVG', '20151213_THWIndex_AVG', '20151213_Bar_AVG', '20151213_Rain_AVG', '20151213_RainRate_AVG', '20151213_SolarRad_AVG', '20151213_SolarEnergy_AVG', '20151213_HiSolarRad_AVG', '20151213_UVIndex_AVG', '20151213_UVDose_AVG', '20151213_HiUV_AVG', '20151213_HeatD_D_AVG', '20151213_Cool_DD_AVG', '20151213_InTemp_AVG', '20151213_InHum_AVG', '20151213_InDew_AVG', '20151213_InHeat_AVG', '20151213_InEMC_AVG', '20151213_InAirDensity_AVG', '20151213_ET_AVG', '20151213_WindSamp_AVG', '20151213_ISSRecept_AVG', '20151214_TempOut_AVG', '20151214_HiTemp_AVG', '20151214_LowTemp_AVG', '20151214_OutHum_AVG', '20151214_DewPt_AVG', '20151214_WindSpeed_AVG', '20151214_WindRun_AVG', '20151214_HiSpeed_AVG

In [85]:
def get_machine_metric(machine, file_csv):
    preprocessing = Preprocess(file_name=file_csv, type_file=TypeFileEnum.CSV)
    preprocessing.read_file(
        transform=TransformEnum.PASS,
        standard_scale=StandardScaleEnum.BASIC,
    )
    preprocessing.build_train_and_test()
    x_train, y_train = preprocessing.get_train()

    # machine = RF_Prediction(n_estimators=1000)
    machine.training(x_train=x_train, y_train=y_train)

    x_test, y_test = preprocessing.get_test()
    y_predicted = machine.prediction(x_test=x_test)

    metric = ErrorMetric(y_predicted=y_predicted, y_test=y_test, x_test=x_test)
    metric.calculate_metric_prediction()
    return (
        metric.get_metric(metric=MetricEnum.MEAN_ABSOLUTE_PERCENTAGE_ERROR),
        metric.get_metric(metric=MetricEnum.ROOT_MEAN_SQUARED_ERROR),
        metric.get_metric(metric=MetricEnum.R2_SCORE),
    )

In [92]:
import itertools

columns_to_iterate = columns_no_ndvi[:-1]
column_y = columns_no_ndvi[-1]

print(columns_to_iterate, column_y)

columns_selection = [
    x for x in itertools.product([True, False], repeat=len(columns_to_iterate))
][:-1]
mape = 1
best = []
metrics = []
for columns in columns_selection:
    selected = np.array(columns_to_iterate)[np.array(columns)].tolist()
    selected.append(column_y)
    file_csv = "test_selection_characteristic.csv"
    df[selected].to_csv(file_csv, index=False)
    new_mape, rsme, r2 = get_machine_metric(
        machine=RF_Prediction(n_estimators=1000), file_csv=file_csv
    )
    metrics.append([",".join(selected), new_mape, rsme,r2])
    if new_mape < mape:
        mape = new_mape
        best = selected

print(mape, best, metrics)

['TGW', 'HI', 'BM', 'CT_UAV_1', 'CT_UAV_2', 'CT_UAV_3', 'CT_UAV_4'] YLD
0.035229980052581135 ['HI', 'BM', 'CT_UAV_1', 'YLD'] [['TGW,HI,BM,CT_UAV_1,CT_UAV_2,CT_UAV_3,CT_UAV_4,YLD', 0.03984041533654076, 37.24790980226826, 0.8559581445349906], ['TGW,HI,BM,CT_UAV_1,CT_UAV_2,CT_UAV_3,YLD', 0.03890828540696177, 35.510000752527255, 0.8690859507242579], ['TGW,HI,BM,CT_UAV_1,CT_UAV_2,CT_UAV_4,YLD', 0.03949102774884862, 38.09609789890206, 0.8493233758842239], ['TGW,HI,BM,CT_UAV_1,CT_UAV_2,YLD', 0.038934387171937465, 37.054073919881965, 0.8574534140442173], ['TGW,HI,BM,CT_UAV_1,CT_UAV_3,CT_UAV_4,YLD', 0.039183627447690356, 37.11984662357708, 0.8569469112769359], ['TGW,HI,BM,CT_UAV_1,CT_UAV_3,YLD', 0.03785530647114226, 34.98472850142707, 0.8729303279754508], ['TGW,HI,BM,CT_UAV_1,CT_UAV_4,YLD', 0.038696711795041666, 37.57082290359949, 0.8534498362848815], ['TGW,HI,BM,CT_UAV_1,YLD', 0.036794383520703335, 35.01335255625817, 0.8727223093690163], ['TGW,HI,BM,CT_UAV_2,CT_UAV_3,CT_UAV_4,YLD', 0.040666849